# 00_Project_Setup_and_Data_Audit.ipynb

Audit real Paderborn data, report corrected 6203 frequency resolution, and freeze the shared recording-level split.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/pinn-motor-fault-code-v2')]
PROJECT_ROOT = next((path.resolve() for path in ROOT_CANDIDATES if (path / 'src' / 'pinn_motor_fault').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate the repository source tree. Set the notebook working directory to the project root.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
REPORTS_DIR = PROJECT_ROOT / 'reports'
if os.environ.get('PINN_PADERBORN_DATA'):
    DATA_DIR = Path(os.environ['PINN_PADERBORN_DATA']).expanduser().resolve()
else:
    KAGGLE_DATA_DIR = Path('/kaggle/input/paderborn')
    LOCAL_DATA_DIR = PROJECT_ROOT / 'data' / 'paderborn' / 'extracted'
    DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.is_dir() else LOCAL_DATA_DIR
print('Project root:', PROJECT_ROOT)
print('Paderborn data:', DATA_DIR)


In [ ]:
from pinn_motor_fault.features import BearingPhysics, PhysicsFeatureExtractor
from pinn_motor_fault.paderborn import MODALITY_SAMPLE_RATES_HZ, load_paderborn_windows_with_metadata
from pinn_motor_fault.train import assert_group_split_no_overlap, stratified_group_split

WINDOW_SIZE = 4096
STRIDE = 2048
MAX_WINDOWS_PER_RECORDING = 16
SEED = 17
physics = BearingPhysics(sample_rate_hz=MODALITY_SAMPLE_RATES_HZ['vibration'])
print('6203 characteristic-frequency ratios:', {
    'BPFO': physics.bpfo_ratio, 'BPFI': physics.bpfi_ratio,
    'BSF': physics.bsf_ratio, 'FTF': physics.ftf_ratio,
})
print('At 25 Hz shaft speed:', physics.characteristic_frequencies(25.0))
print('Generic vibration FFT resolution:', physics.frequency_resolution_hz(WINDOW_SIZE), 'Hz')


## Real Paderborn audit and shared vibration split
This notebook loads actual MAT recordings and saves the exact vibration recording split used by M1 and M3-M6. Loading failures are allowed to propagate; no synthetic replacement is used.

In [ ]:
assert DATA_DIR.is_dir(), f'Paderborn data directory does not exist: {DATA_DIR}'
loaded = load_paderborn_windows_with_metadata(
    DATA_DIR, window_size=WINDOW_SIZE, stride=STRIDE,
    signal_preference='vibration', max_windows_per_file=MAX_WINDOWS_PER_RECORDING,
    require_modality_match=True,
)
assert loaded.modality == 'vibration'
assert loaded.windows.size > 0 and len(loaded.sources) == loaded.labels.size
assert all(Path(source).suffix.lower() == '.mat' for source in loaded.sources), 'Expected real Paderborn MAT recordings.'
data_root = DATA_DIR.resolve()
assert all(Path(source).resolve().is_relative_to(data_root) for source in loaded.sources), 'Loaded recordings must be under DATA_DIR.'
train_indices, test_indices = stratified_group_split(
    loaded.labels, loaded.sources, test_fraction=0.25, seed=SEED,
)
assert_group_split_no_overlap(loaded.sources, train_indices, test_indices)
recordings = sorted(set(str(Path(source)) for source in loaded.sources))
manifest = {
    'dataset': 'real_paderborn', 'modality': loaded.modality,
    'window_size': WINDOW_SIZE, 'stride': STRIDE,
    'sampling_rate_hz': MODALITY_SAMPLE_RATES_HZ[loaded.modality],
    'seed': SEED, 'test_fraction': 0.25,
    'strategy': 'stratified_group_by_mat_file',
    'source_recordings': recordings,
    'train_files': sorted(set(loaded.sources[i] for i in train_indices)),
    'test_files': sorted(set(loaded.sources[i] for i in test_indices)),
    'window_count': int(loaded.labels.size),
    'skipped_recordings': loaded.skipped,
}
shared_dir = REPORTS_DIR / 'shared'
shared_dir.mkdir(parents=True, exist_ok=True)
manifest_path = shared_dir / 'split_manifest.json'
if manifest_path.exists():
    existing = json.loads(manifest_path.read_text(encoding='utf-8'))
    if existing != manifest:
        raise FileExistsError(f'Existing shared split differs; refusing to replace it: {manifest_path}')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps({
    'modality': loaded.modality, 'windows': int(loaded.labels.size),
    'recordings': len(recordings), 'train_recordings': len(manifest['train_files']),
    'test_recordings': len(manifest['test_files']), 'skipped': len(loaded.skipped),
    'manifest': str(manifest_path),
}, indent=2))


## Frequency-resolution report
The 4096-sample window remains the generic baseline. The longer sizes below are diagnostic-only resolution scenarios, not a change to the generic experiment. Zero-padding changes displayed bin spacing but does not increase true resolution, which is determined by the unpadded observation duration.

In [ ]:
window_sizes = (4096, 8192, 16384, 32768)
resolution_report = {
    modality: [
        {
            'window_size_samples': size,
            'duration_seconds': size / rate,
            'true_fft_resolution_hz': rate / size,
            'purpose': 'generic_baseline' if size == 4096 else 'diagnostic_investigation_only',
        }
        for size in window_sizes
    ]
    for modality, rate in MODALITY_SAMPLE_RATES_HZ.items()
}
resolution_path = shared_dir / 'frequency_resolution.json'
resolution_path.write_text(json.dumps(resolution_report, indent=2), encoding='utf-8')
print(json.dumps(resolution_report, indent=2))


## Explicit synthetic smoke test (not an experiment)
Set the flag only to check feature extraction in a test environment without Paderborn data. This section never feeds synthetic samples to an experiment or writes experiment results.

In [ ]:
RUN_SYNTHETIC_SMOKE_TEST = False
if RUN_SYNTHETIC_SMOKE_TEST:
    from pinn_motor_fault.train import make_synthetic_dataset
    smoke_windows, smoke_labels, smoke_sources = make_synthetic_dataset(samples_per_class=2, window_size=WINDOW_SIZE)
    smoke_batch = PhysicsFeatureExtractor(feature_group='generic').transform(smoke_windows, smoke_sources)
    assert smoke_batch.features.shape == (8, 19)
    assert smoke_batch.physics_targets.shape == (8, 4)
    print('Synthetic feature-extraction smoke test passed; no experiment was run.')
